# 05. Linear Regression (OLS) Model
**Crop Yield Prediction and Forecasting Using Agricultural Data and Machine Learning**  
*Author: Aryan Puranik*

This notebook implements the baseline **Ordinary Least Squares (OLS) Linear Regression** model for crop yield prediction ($kg/ha$).

### Objectives
1. **Model Pipeline:** Construct a leak-free scikit-learn pipeline combining numeric scaling (`StandardScaler`), categorical encoding (`OneHotEncoder`), and median imputation.
2. **Selected Predictor Inputs:** Load the standardized feature set vetted in Notebook 04 (`data/processed/selected_features.json`), including agronomic variables, weather anomalies, and historical lags.
3. **Dual-Cohort Validation:** Evaluate predictive accuracy on the unseen 2012–2014 validation set across both the **Complete History Cohort** ($N=9,966$) and the **Full Validation Cohort** ($N=11,345$).
4. **Comprehensive Evaluation Metrics:** Compute MAE, RMSE, $R^2$, MAP (Mean Absolute Percentage Error), and WAPE.
5. **Binned Confusion Matrix:** Discretize continuous actual and predicted yields into 5 ordered quintile tiers to evaluate categorical ranking consistency.
6. **Model Artifact Export:** Save predictions, metrics, confusion matrix, and fitted pipeline to establish Model 1 on the comparative leaderboard.


## 1. Setup, Imports, and Data Loading


In [2]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report, accuracy_score
)

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "model_evaluation"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load engineered splits
train_df = pd.read_csv(PROCESSED_DIR / "train_engineered.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_engineered.csv")

# Load selected features configuration from Notebook 04
with open(PROCESSED_DIR / "selected_features.json", "r") as f:
    config = json.load(f)

numerical_features = config["numerical_features"]
categorical_features = config["categorical_features"]
target_col = config["target"]

print(f"Training data:   {train_df.shape[0]:,} rows | Years: {train_df['Year'].min()}-{train_df['Year'].max()}")
print(f"Validation data: {val_df.shape[0]:,} rows | Years: {val_df['Year'].min()}-{val_df['Year'].max()}")
print(f"Numerical features ({len(numerical_features)}):", numerical_features)
print(f"Categorical features ({len(categorical_features)}):", categorical_features)
print(f"Target variable: '{target_col}'")


Training data:   176,194 rows | Years: 1966-2011
Validation data: 11,345 rows | Years: 2012-2014
Numerical features (17): ['Year', 'Temperature', 'Rainfall', 'Humidity', 'pH', 'Nitrogen', 'Phosphorus', 'Potassium', 'Pesticide', 'Yield_Lag1', 'Yield_Rolling3', 'Yield_Rolling3_Std', 'Yield_Change_1to5', 'History_Lag1_Available', 'History_3yr_Complete', 'Temperature_Anomaly', 'Rainfall_Anomaly']
Categorical features (3): ['Region', 'Crop', 'Soil Type']
Target variable: 'Yield'


## 2. Pipeline Construction and Model Fitting

We construct an encapsulated pipeline:
- **Numerical Features:** Imputed with median and standardized using `StandardScaler`.
- **Categorical Features:** Imputed with most frequent value and one-hot encoded (`OneHotEncoder(handle_unknown='ignore')`).
- **Estimator:** Ordinary Least Squares `LinearRegression()`.

All transformations are fitted strictly on training observations (1966–2011).


In [4]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Training
t0 = time.time()
feature_cols = numerical_features + categorical_features
X_train = train_df[feature_cols]
y_train = train_df[target_col]

linear_pipeline.fit(X_train, y_train)
train_time = time.time() - t0

print(f"Linear Regression pipeline trained successfully in {train_time:.2f} seconds.")


Linear Regression pipeline trained successfully in 6.66 seconds.


## 3. Validation Inference & Comprehensive Metrics

We evaluate on the unseen **2012–2014 Validation Period** across two cohorts:
1. **Complete History Cohort ($N=9,966$):** Rows where all 3 previous years of yield are present.
2. **Full Validation Cohort ($N=11,345$):** All validation observations.

Metrics computed:
- **MAE (kg/ha):** $\frac{1}{N}\sum |y_i - \hat{y}_i|$
- **RMSE (kg/ha):** $\sqrt{\frac{1}{N}\sum (y_i - \hat{y}_i)^2}$
- **$R^2$:** Proportion of variance explained
- **MAP (%):** Mean Absolute Percentage Error (evaluated on $y_i \ge 10$)
- **WAPE (%):** Weighted Absolute Percentage Error $\frac{\sum |y_i - \hat{y}_i|}{\sum y_i} \times 100$


In [6]:
# Inference
X_val = val_df[feature_cols]
y_val = val_df[target_col].values
val_preds = linear_pipeline.predict(X_val)

def compute_metrics(y_true, y_pred, label):
    err = y_true - y_pred
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    
    valid_mask = y_true >= 10.0
    map_err = float(100.0 * np.mean(np.abs(err[valid_mask] / y_true[valid_mask])))
    wape = float(100.0 * np.sum(np.abs(err)) / np.sum(y_true))
    
    return {
        "Cohort": label,
        "Rows": len(y_true),
        "MAE (kg/ha)": round(mae, 2),
        "RMSE (kg/ha)": round(rmse, 2),
        "R2": round(r2, 4),
        "MAP (%)": round(map_err, 2),
        "WAPE (%)": round(wape, 2)
    }

# 1. Full Validation Cohort
full_metrics = compute_metrics(y_val, val_preds, "Full Validation Cohort")

# 2. Complete History Cohort
comp_mask = val_df["History_3yr_Complete"] == 1
comp_metrics = compute_metrics(y_val[comp_mask], val_preds[comp_mask], "Complete History Cohort (N=9,966)")

# Benchmark comparison table
benchmark_comparison = pd.DataFrame([
    {
        "Model / Strategy": "Naive 3-Year Mean Benchmark",
        "Cohort": "Complete History (N=9,966)",
        "MAE (kg/ha)": 314.66,
        "RMSE (kg/ha)": 562.34,
        "R2": 0.8944,
        "MAP (%)": 29.39,
        "WAPE (%)": 28.52
    },
    {
        "Model / Strategy": "Linear Regression (OLS)",
        "Cohort": comp_metrics["Cohort"],
        "MAE (kg/ha)": comp_metrics["MAE (kg/ha)"],
        "RMSE (kg/ha)": comp_metrics["RMSE (kg/ha)"],
        "R2": comp_metrics["R2"],
        "MAP (%)": comp_metrics["MAP (%)"],
        "WAPE (%)": comp_metrics["WAPE (%)"]
    },
    {
        "Model / Strategy": "Linear Regression (OLS)",
        "Cohort": full_metrics["Cohort"],
        "MAE (kg/ha)": full_metrics["MAE (kg/ha)"],
        "RMSE (kg/ha)": full_metrics["RMSE (kg/ha)"],
        "R2": full_metrics["R2"],
        "MAP (%)": full_metrics["MAP (%)"],
        "WAPE (%)": full_metrics["WAPE (%)"]
    }
])

print("=== LINEAR REGRESSION VALIDATION PERFORMANCE ===")
display(benchmark_comparison)


=== LINEAR REGRESSION VALIDATION PERFORMANCE ===
              Model / Strategy  ...  WAPE (%)
0  Naive 3-Year Mean Benchmark  ...     28.52
1      Linear Regression (OLS)  ...     19.25
2      Linear Regression (OLS)  ...     21.83

[3 rows x 7 columns]


## 4. Binned Confusion Matrix for Continuous Yield

To evaluate how accurately Linear Regression classifies crops into relative yield tiers, we discretize actual and predicted yields into 5 ordered quintile bins based on training distribution:
- **Tier 1 (Very Low):** 0 – 20th percentile
- **Tier 2 (Low):** 20th – 40th percentile
- **Tier 3 (Medium):** 40th – 60th percentile
- **Tier 4 (High):** 60th – 80th percentile
- **Tier 5 (Very High):** 80th – 100th percentile


In [8]:
# Determine quantile thresholds from training data
quantiles = [0.2, 0.4, 0.6, 0.8]
q_cuts = list(np.quantile(train_df[target_col].values, quantiles))
bin_edges = [-np.inf] + q_cuts + [np.inf]
tier_labels = ["Very Low", "Low", "Medium", "High", "Very High"]

y_true_tiers = pd.cut(y_val, bins=bin_edges, labels=tier_labels)
y_pred_tiers = pd.cut(val_preds, bins=bin_edges, labels=tier_labels)

# Compute confusion matrix
cm = confusion_matrix(y_true_tiers, y_pred_tiers, labels=tier_labels)
tier_acc = accuracy_score(y_true_tiers, y_pred_tiers)

cm_df = pd.DataFrame(cm, index=[f"Actual {t}" for t in tier_labels], columns=[f"Pred {t}" for t in tier_labels])
print(f"Yield Tier Classification Accuracy: {tier_acc * 100:.2f}%")
print()
display(cm_df)

# Plot confusion matrix heatmap
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="YlGnBu", cbar=True, ax=ax)
ax.set_title(f"Linear Regression: Binned Yield Confusion Matrix (Accuracy: {tier_acc*100:.1f}%)")
plt.tight_layout()
plt.show()


Yield Tier Classification Accuracy: 63.68%

                  Pred Very Low  Pred Low  ...  Pred High  Pred Very High
Actual Very Low             440       452  ...         53               5
Actual Low                  124       735  ...        167              22
Actual Medium                 9       192  ...        639              68
Actual High                   4        36  ...       1702             423
Actual Very High              0         5  ...        507            3253

[5 rows x 5 columns]


## 5. Residual Diagnostics and Error Visualizations

We assess residual normality, heteroskedasticity, and prediction parity:
1. **Actual vs. Predicted Yield:** Closer alignment to the $45^\circ$ dashed line indicates lower bias.
2. **Residual Distribution:** Distribution of $(y - \hat{y})$ centered around zero.
3. **Residuals vs. Predicted:** Checks for variance expansion at higher yield scales.


In [10]:
residuals = y_val - val_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_val, val_preds, alpha=0.15, color="steelblue", s=10)
max_val = max(np.max(y_val), np.max(val_preds))
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Parity (y = ŷ)")
axes[0].set_title("Actual vs. Predicted Yield")
axes[0].set_xlabel("Actual Yield (kg/ha)")
axes[0].set_ylabel("Predicted Yield (kg/ha)")
axes[0].set_xlim(0, max_val * 1.05)
axes[0].set_ylim(0, max_val * 1.05)
axes[0].legend()

# 2. Residual Distribution
sns.histplot(residuals, bins=60, kde=True, ax=axes[1], color="teal")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title(f"Residual Distribution (Mean: {np.mean(residuals):.2f})")
axes[1].set_xlabel("Residual: Actual - Predicted (kg/ha)")
axes[1].set_xlim(-2500, 2500)

# 3. Residuals vs Predicted
axes[2].scatter(val_preds, residuals, alpha=0.15, color="darkorange", s=10)
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set_title("Residuals vs. Predicted Yield")
axes[2].set_xlabel("Predicted Yield (kg/ha)")
axes[2].set_ylabel("Residual (kg/ha)")
axes[2].set_ylim(-4000, 4000)

plt.tight_layout()
plt.show()


## 6. Crop-Wise Performance Breakdown

Agricultural yield scales vary significantly across crops (e.g., Sugarcane ~10,000 kg/ha vs. Linseed ~500 kg/ha). We inspect crop-specific errors to evaluate localized model behavior.


In [12]:
val_analysis_df = val_df[["Region", "Crop", "Year", target_col]].copy()
val_analysis_df["Pred_Yield"] = val_preds
val_analysis_df["Error"] = np.abs(val_analysis_df[target_col] - val_preds)

crop_summary = val_analysis_df.groupby("Crop").apply(lambda g: pd.Series({
    "Samples": len(g),
    "Mean Actual (kg/ha)": g[target_col].mean(),
    "MAE (kg/ha)": mean_absolute_error(g[target_col], g["Pred_Yield"]),
    "RMSE (kg/ha)": np.sqrt(mean_squared_error(g[target_col], g["Pred_Yield"])),
    "R2": r2_score(g[target_col], g["Pred_Yield"]),
    "WAPE (%)": 100.0 * np.sum(np.abs(g[target_col] - g["Pred_Yield"])) / np.sum(g[target_col])
})).reset_index().sort_values("Samples", ascending=False)

print("Crop-Wise Validation Performance:")
display(crop_summary.round(2))


Crop-Wise Validation Performance:
                  Crop  Samples  ...    R2  WAPE (%)
12                Rice    818.0  ...  0.78     13.08
7                Maize    805.0  ...  0.64     22.74
2             Chickpea    749.0  ...  0.28     25.93
10           Pigeonpea    745.0  ...  0.46     26.24
8         Minor Pulses    742.0  ...  0.02     33.77
20               Wheat    740.0  ...  0.67     17.42
14             Sesamum    737.0  ...  0.43     33.79
18           Sugarcane    733.0  ...  0.53     19.34
5            Groundnut    670.0  ...  0.62     18.75
11  Rapeseed & Mustard    660.0  ...  0.51     23.60
15    Sorghum (Kharif)    547.0  ...  0.37     28.60
9         Pearl Millet    529.0  ...  0.47     25.06
0               Barley    434.0  ...  0.55     21.14
3               Cotton    392.0  ...  0.04     43.39
17             Soybean    379.0  ...  0.13     31.73
19           Sunflower    370.0  ...  0.40     24.10
6              Linseed    360.0  ...  0.53     25.09
1           

## 7. Model Artifact Export and Leaderboard Initialization

We save all evaluation outputs to disk for downstream comparison:
1. **Model Pipeline:** `models/linear_regression_pipeline.joblib`
2. **Metrics Summary:** `reports/model_evaluation/linear_regression_metrics.json`
3. **Validation Predictions:** `reports/model_evaluation/linear_regression_val_predictions.csv`
4. **Confusion Matrix:** `reports/model_evaluation/linear_regression_confusion_matrix.csv`
5. **Leaderboard Update:** `reports/model_evaluation/model_comparison_leaderboard.csv`


In [14]:
# 1. Save fitted pipeline
pipeline_path = MODELS_DIR / "linear_regression_pipeline.joblib"
joblib.dump(linear_pipeline, pipeline_path)
print(f"Saved pipeline to {pipeline_path}")

# 2. Save metrics dictionary
metrics_payload = {
    "model": "Linear Regression (OLS)",
    "training_time_seconds": round(train_time, 2),
    "tier_classification_accuracy": round(tier_acc, 4),
    "full_validation": full_metrics,
    "complete_history_validation": comp_metrics
}
metrics_path = REPORTS_DIR / "linear_regression_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Saved metrics to {metrics_path}")

# 3. Save validation predictions with tiers
val_predictions_export = val_df[["Region", "Crop", "Year", target_col]].copy()
val_predictions_export["Predicted_Yield"] = val_preds
val_predictions_export["Residual"] = residuals
val_predictions_export["Actual_Tier"] = y_true_tiers
val_predictions_export["Predicted_Tier"] = y_pred_tiers

pred_path = REPORTS_DIR / "linear_regression_val_predictions.csv"
val_predictions_export.to_csv(pred_path, index=False)
print(f"Saved predictions to {pred_path}")

# 4. Save confusion matrix CSV
cm_path = REPORTS_DIR / "linear_regression_confusion_matrix.csv"
cm_df.to_csv(cm_path)
print(f"Saved confusion matrix to {cm_path}")

# 5. Initialize or update model comparison leaderboard
leaderboard_row = {
    "Model": "Linear Regression (OLS)",
    "Full_Val_MAE": full_metrics["MAE (kg/ha)"],
    "Full_Val_RMSE": full_metrics["RMSE (kg/ha)"],
    "Full_Val_R2": full_metrics["R2"],
    "Full_Val_MAP": full_metrics["MAP (%)"],
    "Complete_Hist_MAE": comp_metrics["MAE (kg/ha)"],
    "Complete_Hist_RMSE": comp_metrics["RMSE (kg/ha)"],
    "Complete_Hist_R2": comp_metrics["R2"],
    "Complete_Hist_MAP": comp_metrics["MAP (%)"],
    "Tier_Accuracy": round(tier_acc * 100, 2)
}

leaderboard_path = REPORTS_DIR / "model_comparison_leaderboard.csv"
if leaderboard_path.exists():
    leaderboard_df = pd.read_csv(leaderboard_path)
    leaderboard_df = leaderboard_df[leaderboard_df["Model"] != "Linear Regression (OLS)"]
    leaderboard_df = pd.concat([leaderboard_df, pd.DataFrame([leaderboard_row])], ignore_index=True)
else:
    leaderboard_df = pd.DataFrame([leaderboard_row])

leaderboard_df.to_csv(leaderboard_path, index=False)
print(f"Updated Leaderboard at {leaderboard_path}:")
display(leaderboard_df)


Saved pipeline to D:\crop_yield_prediction\models\linear_regression_pipeline.joblib
Saved metrics to D:\crop_yield_prediction\reports\model_evaluation\linear_regression_metrics.json
Saved predictions to D:\crop_yield_prediction\reports\model_evaluation\linear_regression_val_predictions.csv
Saved confusion matrix to D:\crop_yield_prediction\reports\model_evaluation\linear_regression_confusion_matrix.csv
Updated Leaderboard at D:\crop_yield_prediction\reports\model_evaluation\model_comparison_leaderboard.csv:
                     Model  Full_Val_MAE  ...  Complete_Hist_MAP  Tier_Accuracy
0  Linear Regression (OLS)        356.04  ...              33.45          63.68

[1 rows x 10 columns]
